# Проект модуля. Нейросеть для автодополнения текстов

## Бизнес-контекст и задача проекта
Соцсетевое приложение, где пользователи постят короткие тексты. В продукте стоит задача — добавить возможность автодополнения текстов. Необходимо оздать модель, которую можно запускать на мобильных устройствах. Для смартфонов есть значительные требования по оперативной памяти и скорости работы, так что важна легковесность модели. 

## Формальная задача
Разработчики предоставили вам датасет с короткими текстовыми постами. 
### Поэтапное описание задачи:
- Взять датасет от разработчиков, очистить его, подготовить для обучения модели.
- Реализовать и обучить модель на основе рекуррентных нейронных сетей.
- Замерить качество разработанной и обученной модели.
- Взять более «тяжёлую» предобученную модель из Transformers и замерить её качество.
- Проанализировать результаты и дать рекомендации разработчикам: стоит ли использовать лёгкую модель или лучше постараться поработать с ограничениями по памяти и использовать большую предобученную.

## Загрузка библиотек

In [1]:
#Автоматическая перезагрузка при изменениях
%load_ext autoreload
%autoreload 2

In [2]:
# Для MPS рекомендуется включить fallback на CPU для неподдерживаемых операций
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [21]:
from src.data_utils import load_and_clear_data, load_or_tokenize, create_sequences, create_data_split, build_vocab
from src.constants import BATCH_SIZE
from src.next_token_dataset import NextTokenDataset, collate_fn
from src.lstm_model import LSTMLanguageModel
from src.lstm_train import train_model
from src.lstm_utils import save_model_weight
from src.eval_transformer_pipeline import evaluate_transformer, test_transformer
from src.constants import PAD_TOKEN

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

## Этап 1. Сбор и подготовка данных

In [4]:
texts = load_and_clear_data()

✅ Загружено 1600000 текстов, после очистки осталось 1597104 текстов.
Примеры очищенных текстов: ["a that's a bummer. you shoulda got david carr of third day to do it. d", "is upset that he can't update his facebook by texting it... and might cry as a result school today also. blah!", 'i dived many times for the ball. managed to save 50 the rest go out of bounds', 'my whole body feels itchy and like its on fire', "no, it's not behaving at all. i'm mad. why am i here? because i can't see you all over there."]


In [5]:
tokenized_texts = load_or_tokenize(texts)

Загрузка токенизированных данных из data/tokenized_texts.pkl...


In [6]:
X, Y = create_sequences(tokenized_texts)

In [7]:
display(X[0:1])
display(Y[0:1])

[['<BOS>',
  'a',
  'that',
  "'s",
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd']]

[['a',
  'that',
  "'s",
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd',
  '<EOS>']]

In [8]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = create_data_split(X, Y) 

Train: 1277683, Val: 159710, Test: 159711


In [9]:
# Создание словаря
word2idx, idx2word = build_vocab(tokenized_texts)

vocab_size = len(word2idx)

Всего уникальных токенов: 369896
Топ-10 частых: [('<BOS>', 1597104), ('<EOS>', 1597104), ('i', 949180), ('!', 916243), ('.', 797560), ('to', 564942), ('the', 521098), (',', 483127), ('a', 383113), ('my', 315823)]
✅ Размер словаря: 19998 (ограничено с 369896)


In [10]:
# Создание датасетов
train_dataset = NextTokenDataset(X_train, Y_train, word2idx)
val_dataset = NextTokenDataset(X_val, Y_val, word2idx)
test_dataset = NextTokenDataset(X_test, Y_test, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

In [11]:
lengths = [len(tokens) for tokens in tokenized_texts]
print(f"Средняя длина: {sum(lengths) / len(lengths):.1f}")
print(f"Медиана: {sorted(lengths)[len(lengths)//2]}")
print(f"99-й перцентиль: {sorted(lengths)[int(len(lengths)*0.99)]}")

Средняя длина: 16.8
Медиана: 16
99-й перцентиль: 35


## Этап 2. Объявление модели LSTM

In [12]:
device = (
        "cuda" if torch.cuda.is_available() else
        "mps" if torch.backends.mps.is_available() else
        "cpu"
    )
print(f"Using device: {device}")

print("torch.backends.mps.is_available:", torch.backends.mps.is_available())
print("torch.backends.mps.is_built:", torch.backends.mps.is_built())

Using device: cuda
torch.backends.mps.is_available: False
torch.backends.mps.is_built: False


In [13]:
model = LSTMLanguageModel(vocab_size, word2idx, idx2word).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD_TOKEN])
optimizer = optim.Adam(model.parameters(), lr=0.0005)

## Этап 3. Тренировка модели LSTM

In [14]:
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, idx2word, word2idx)

Epoch 1/10 [Train]:   0%|          | 2/2496 [00:01<23:35,  1.76it/s, loss=9.9102]


[Epoch 1, Batch 0]
  Allocated: 1.47 GB
  Reserved: 5.89 GB


Calculating ROUGE scores:   0%|          | 0/312 [00:00<?, ?it/s]


🥑Примеры генерации модели (LSTM)

[Пример 1]
Вход (75%): will take your mind off everything ! just put in the crate like . he
Цель (25%): wo n't even remember it later .
Предсказание: need to go to go watching the
Полное предложение: will take your mind off everything ! just put in the crate like . he need to go to go watching the

[Пример 2]
Вход (75%): does n't know what to do ... day
Цель (25%): is looking grim
Предсказание: there ! !
Полное предложение: does n't know what to do ... day there ! !

[Пример 3]
Вход (75%): yay ! thanks ! have a
Цель (25%): restful evening !
Предсказание: way to them
Полное предложение: yay ! thanks ! have a way to them

[Пример 4]
Вход (75%): gon na help with kids this friday
Цель (25%): ! whee !
Предсказание: 
Полное предложение: gon na help with kids this friday

[Пример 5]
Вход (75%): wan na go to
Цель (25%): disneyland .
Предсказание: follow that
Полное предложение: wan na go to follow that

[Пример 6]
Вход (75%): hope your boyfriend buys it
Цель 

Calculating ROUGE scores: 100%|██████████| 312/312 [07:04<00:00,  1.36s/it]


...


[Epoch 10, Batch 0]
  Allocated: 1.47 GB
  Reserved: 7.23 GB


Calculating ROUGE scores:   0%|          | 0/312 [00:00<?, ?it/s]


🥑Примеры генерации модели (LSTM)

[Пример 1]
Вход (75%): will take your mind off everything ! just put in the crate like . he
Цель (25%): wo n't even remember it later .
Предсказание: 's leaving us .
Полное предложение: will take your mind off everything ! just put in the crate like . he 's leaving us .

[Пример 2]
Вход (75%): does n't know what to do ... day
Цель (25%): is looking grim
Предсказание: to be done
Полное предложение: does n't know what to do ... day to be done

[Пример 3]
Вход (75%): yay ! thanks ! have a
Цель (25%): restful evening !
Предсказание: safe recipe !
Полное предложение: yay ! thanks ! have a safe recipe !

[Пример 4]
Вход (75%): gon na help with kids this friday
Цель (25%): ! whee !
Предсказание: ...
Полное предложение: gon na help with kids this friday ...

[Пример 5]
Вход (75%): wan na go to
Цель (25%): disneyland .
Предсказание: school tonight
Полное предложение: wan na go to school tonight

[Пример 6]
Вход (75%): hope your boyfriend buys it
Цель (25%): fo

Calculating ROUGE scores: 100%|██████████| 312/312 [07:20<00:00,  1.41s/it]

Epoch 10, Train Loss: 4.7137, Val loss: 4.6498
Val ROUGE-1: 0.0499, Val ROUGE-2: 0.0070 


In [15]:
save_model_weight(model)

✅ Веса модели сохранены в models/lstm_model_weights.pth


### Этап 4. Использование предобученного трансформера

In [ ]:
model, results = evaluate_transformer(val_loader, idx2word, word2idx, device=device)


Оценка на валидационной выборке (DISTILGPT2)


Val: Processing batches:   0%|          | 0/312 [00:00<?, ?it/s]


[Пример 1]
Вход (75%): will take your mind off everything ! just put in the crate like . he
Цель (25%): wo n't even remember it later .
Предсказание: 'll be back in the car !
Полное предложение: will take your mind off everything ! just put in the crate like . he'll be back in the car !

[Пример 2]
Вход (75%): does n't know what to do ... day
Цель (25%): is looking grim
Предсказание: one.�
Полное предложение: does n't know what to do ... day one.�

[Пример 3]
Вход (75%): yay ! thanks ! have a
Цель (25%): restful evening !
Предсказание: look at my
Полное предложение: yay ! thanks ! have a look at my

[Пример 4]
Вход (75%): gon na help with kids this friday
Цель (25%): ! whee !
Предсказание: in NYC?
Полное предложение: gon na help with kids this friday in NYC?

[Пример 5]
Вход (75%): wan na go to
Цель (25%): disneyland .
Предсказание: sleep,
Полное предложение: wan na go to sleep,

[Пример 6]
Вход (75%): hope your boyfriend buys it
Цель (25%): for you xoxo
Предсказание: . He doesn
Полно

Val: Processing batches: 100%|██████████| 312/312 [1:28:20<00:00, 16.99s/it]



Метрики ROUGE на Val (DISTILGPT2)
ROUGE-1 (F1): 0.0522
ROUGE-2 (F1): 0.0048


In [26]:
test_transformer(model, test_loader, idx2word, word2idx, device)


ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ (DISTILGPT2)


Test: Processing batches:   0%|          | 0/312 [00:00<?, ?it/s]


[Пример 1]
Вход (75%): totally love your music , super excited about seeing you guys play at
Цель (25%): lake ny for my bday
Предсказание: a concert or some other
Полное предложение: totally love your music , super excited about seeing you guys play at a concert or some other

[Пример 2]
Вход (75%): am working on it ! 've got some little mo designs 'm working
Цель (25%): on and of course !
Предсказание: on it ! '
Полное предложение: am working on it ! 've got some little mo designs 'm working on it ! '

[Пример 3]
Вход (75%): whaat ? why ? you can download it
Цель (25%): from the internet
Предсказание: for Windows,
Полное предложение: whaat ? why ? you can download it for Windows,

[Пример 4]
Вход (75%): think 'm going to have
Цель (25%): another sleepless night
Предсказание: to take a
Полное предложение: think 'm going to have to take a

[Пример 5]
Вход (75%): thanks and promise you guys wo n't have to kill me thank you and ehhh we
Цель (25%): 'll see how it works out
Предсказание: w

Test: Processing batches: 100%|██████████| 312/312 [1:28:38<00:00, 17.05s/it]



Метрики ROUGE на TEST (DISTILGPT2)
ROUGE-1 (F1): 0.0522
ROUGE-2 (F1): 0.0049


(<src.eval_transformer_pipeline.DistilGPT2Model at 0x7d338c65d690>,
 {'rouge1': np.float64(0.05216672336104809),
  'rouge2': np.float64(0.004891453000953238),
  'rougeL': np.float64(0.05160300157433113),
  'rougeLsum': np.float64(0.05161061786362503)})